In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/raw/Hyderabad/telangana_opendata_gtfs_tgsrtc_08_february_2026")

routes = pd.read_csv(data_path / "routes.txt", dtype=str)
trips = pd.read_csv(data_path / "trips.txt", dtype=str)
stops = pd.read_csv(data_path / "stops.txt", dtype=str)
stop_times = pd.read_csv(data_path / "stop_times.txt", dtype=str)

print("Routes:", routes.shape)
print("Trips:", trips.shape)
print("Stops:", stops.shape)
print("Stop times:", stop_times.shape)

print("\nStop times columns:", stop_times.columns.tolist())

Routes: (1031, 4)
Trips: (44892, 5)
Stops: (5028, 6)
Stop times: (1132807, 6)

Stop times columns: ['trip_id', 'stop_sequence', 'stop_id', 'departure_time', 'arrival_time', 'timepoint']


In [2]:
# Count distinct trips per route
trip_counts = trips.groupby("route_id")["trip_id"].nunique().reset_index(name="total_trips")

# Count distinct stops per route
trip_stops = trips[["route_id", "trip_id"]].merge(
    stop_times[["trip_id", "stop_id"]],
    on="trip_id",
    how="left"
)

stop_counts = trip_stops.groupby("route_id")["stop_id"].nunique().reset_index(name="total_stops")

# Combine route details and features
route_features = routes.merge(trip_counts, on="route_id", how="left")
route_features = route_features.merge(stop_counts, on="route_id", how="left")

print(route_features.head())

  route_id route_long_name agency_id route_type  total_trips  total_stops
0        1               1    TGSRTC          3          305          134
1       10              10    TGSRTC          3          120           67
2  100/299         100/299    TGSRTC          3            5           67
3     100B            100B    TGSRTC          3            1           23
4     100G            100G    TGSRTC          3            6           83


In [3]:
# Convert GTFS time into decimal hours
def gtfs_hour(t):
    try:
        h, m, s = map(int, t.split(":"))
        return h + m / 60 + s / 3600
    except:
        return None

stop_times["arrival_hour"] = stop_times["arrival_time"].apply(gtfs_hour)
stop_times["departure_hour"] = stop_times["departure_time"].apply(gtfs_hour)

# Classify time periods
def time_period(h):
    if pd.isna(h):
        return "Unknown"
    if 5 <= h < 12:
        return "Morning"
    if 12 <= h < 17:
        return "Afternoon"
    if 17 <= h < 21:
        return "Evening"
    return "Night"

stop_times["arrival_period"] = stop_times["arrival_hour"].apply(time_period)
stop_times["departure_period"] = stop_times["departure_hour"].apply(time_period)

# Get first departure of each trip
first_departures = (
    stop_times.sort_values("stop_sequence")
    .drop_duplicates("trip_id")
)

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

print(stop_times[["arrival_hour", "departure_hour", "departure_period"]].head())
print("\nHourly trip counts:")
print(hourly_trips.head())

   arrival_hour  departure_hour departure_period
0     14.166667       14.166667        Afternoon
1     14.207778       14.207778        Afternoon
2     14.276389       14.276389        Afternoon
3     14.374444       14.374444        Afternoon
4     14.492778       14.492778        Afternoon

Hourly trip counts:
  route_id  departure_hour  trip_count
0        1        3.666667           1
1        1        4.083333           1
2        1        4.166667           2
3        1        4.333333           1
4        1        4.750000           1


In [4]:
output_path = Path("../data/processed/hyderabad")
output_path.mkdir(parents=True, exist_ok=True)

# Save route profile and hourly trip counts
route_features.to_csv(output_path / "route_profile.csv", index=False)
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)

# Save time features
stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Hyderabad CSV files saved successfully!")

Hyderabad CSV files saved successfully!


In [5]:
print("Missing values:")
print(route_features.isnull().sum())

print("\nDuplicate route IDs:", route_features["route_id"].duplicated().sum())
print("Duplicate trip IDs:", trips["trip_id"].duplicated().sum())

print("\nMissing arrival hours:", stop_times["arrival_hour"].isna().sum())
print("Missing departure hours:", stop_times["departure_hour"].isna().sum())

Missing values:
route_id           0
route_long_name    0
agency_id          0
route_type         0
total_trips        0
total_stops        0
dtype: int64

Duplicate route IDs: 0
Duplicate trip IDs: 0

Missing arrival hours: 0
Missing departure hours: 0


In [6]:
def gtfs_minute(t):
    try:
        h, m, s = map(int, t.split(":"))
        return m
    except:
        return None

stop_times["arrival_minute"] = stop_times["arrival_time"].apply(gtfs_minute)
stop_times["departure_minute"] = stop_times["departure_time"].apply(gtfs_minute)

stop_times["stop_sequence"] = pd.to_numeric(
    stop_times["stop_sequence"], errors="coerce"
)

first_departures = (
    stop_times.sort_values("stop_sequence")
    .drop_duplicates("trip_id")
)

first_departures["departure_service_hour"] = (
    first_departures["departure_hour"].apply(
        lambda h: int(h) if pd.notna(h) else None
    )
)

trip_hours = first_departures.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

hourly_trips = (
    trip_hours.groupby(["route_id", "departure_service_hour"])["trip_id"]
    .nunique()
    .reset_index(name="trip_count")
)

output_path = Path("../data/processed/hyderabad")
hourly_trips.to_csv(output_path / "hourly_trip_counts.csv", index=False)
stop_times.to_csv(output_path / "time_features.csv", index=False)

print("Hyderabad updated successfully!")

Hyderabad updated successfully!
